tools <br>
say tool is function in each tool we need to give task specific name for that tool <br>
and doc strings such that model will understands what the tool is about <br>
type hints for arguments so that agent will know what type of arguments to be passed <br>


before mcp for agents we have llms, there will be a class for agent , now in the prompt we have to pass user query and tools information dynamically ...for that agent will have the tools info ...when llm gives list of tools and input for those tools the agent will execute those tools with input given by llm... now difficulty comes when we have external tools providers..... suppose we have 100 external tools ...our agent has to connect to those 100 tools ..say those 100 tools are via api of those tools and api endpoints of them changes ....now we have first know the correct api details and manually change the api details and args or anything ...it will be a headache for us to handle .....suppose there are M agents and N tools totally we need M*N connections .....but with mcp ...we can get it done using M+N connections .... agent never takes heacache of tools ....there is a gateway kind of thing called mcp server so our agent will commnuincate to that mcp seerver and mcp server is mainitend by tool providers so if any changes are there they will take care ...... we just need to connect to the mcp server...mcp server has two responsilits it will give list of tool/prompts/resource schema and it will invoke the tools which is said my agent.... to communicate with mcp server they defined a protocal like jscon-rpc so to follow and send req and receive response we need mcp client ....for every single agent there will be a mcp client so in a multi agent system say there are m agents we require m mcp clients and each client will connect to n number of mcp servers as required where n = 0,1,2 ...  this is my understanings .... even say with in organization our team is providing tools ....at that time we can do without mcp like agent class but if we provide tools under mcp server ...it will loosly coupled the agent code no need to bother about tools how they defined and all .....different developers can work on that 

MCP Architecture <br>
1.MCP Host : The Host is the user-facing AI application that end-users interact with directly.The Host’s responsibilities include:Initiating connections to MCP Servers via MCP Clients <br>
2.MCP Client : The Client is a component within the Host application that manages communication with a specific MCP Server. Key characteristics include:Each Client maintains a 1:1 connection with a single Server
so for m agents we will have m clients <br>
3.MCP Server : The Server is an external program or service that exposes capabilities to AI models via the MCP protocol. 


communication flow 

User Interaction: The user interacts with the Host application, expressing an intent or query.

Host Processing: The Host processes the user’s input, potentially using an LLM to understand the request and determine which external capabilities might be needed.

Client Connection: The Host directs its Client component to connect to the appropriate Server(s).

Capability Discovery: The Client queries the Server to discover what capabilities (Tools, Resources, Prompts) it offers.

Capability Invocation: Based on the user’s needs or the LLM’s determination, the Host instructs the Client to invoke specific capabilities from the Server.

Server Execution: The Server executes the requested functionality and returns results to the Client.

Result Integration: The Client relays these results back to the Host, which incorporates them into the context for the LLM or presents them directly to the user.

in simple  <br>
The Host is the main application that contains the Agent/LLM and the MCP Client. <br>
agents which requires tools will be connected to a mcp client, when agent sends it's requirements like which tool to call and it's arguments to client and client sends to server and server executes it and returns output to client and client returns it to agent 

#### Implementing MCP Server

MCP server has two types of communication mechanism or transport modes:

stdio transport → local process communication , without internet in local system

network transport(SSE) → HTTP / WebSocket communication , means over internet

Both use the same MCP protocol (JSON-RPC). Only the transport layer differs.

#### 1. stdio transport (default) — NO IP / PORT

stdio = standard input/output

This means the MCP server talks through the same process pipe, not through the network.

Client process  ←→  stdin/stdout  ←→  MCP server process

example : 

```
mcp.run()

```

this uses  <b> transport="stdio" <b> 

No network involved → so no IP or port. 


In [ ]:
### MCP server implementation using stdio
from mcp.server.fastmcp import FastMCP

# create mcp server instance 

mcp = FastMCP()

# register the tools using @tool decorator

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b

if __name__ == "__main__":
    mcp.run()



What MCP server automatically handles :

It provides endpoints like:

tools/list

tools/call

resources/list

You don’t write these manually.

SDK handles everything.

In [ ]:
### now in the above we have seen we have given just mcp.run() but we didn't give host and port number like fastapi/flask we can't give directly



#### 2. SSE transport (HTTP) — USES IP / PORT

SSE = Server-Sent Events

This uses HTTP network communication.

Client ←→ HTTP ←→ MCP server
         (IP + PORT)



In [ ]:
### mcp server which uses HTTP 

# FastMCP supports HTTP via SSE (Server-Sent Events).
# You must use the async SSE runner.

from mcp.server.fastmcp import FastMCP
import anyio

mcp = FastMCP()

@mcp.tool()
def multiply(a: int, b: int) -> int:
    return a * b

if __name__ == "__main__":
    anyio.run(
        mcp.run_sse_async,
        host="127.0.0.1",
        port=8000,
        mount_path="/sse"
    )
    

#acces url : http://127.0.0.1:8000/sse

### Reload support (like uvicorn --reload)

# FastMCP itself does NOT provide reload.

# Use external reload tools like: uvicorn wrapper (advanced)

# FastMCP internally uses Starlette, so you can mount it:
import uvicorn
from mcp.server.fastmcp import FastMCP

mcp = FastMCP()

app = mcp.sse_app()

if __name__ == "__main__":
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        reload=True
    )



#### web socket mcp server
# websocket_server.py

from mcp.server.fastmcp import FastMCP
import uvicorn

mcp = FastMCP("Weather WS Server")

@mcp.tool()
def get_weather(city: str) -> str:
    return f"Weather in {city} is 30°C"

app = mcp.websocket_app()   # this line creates a FastAPI app with WebSocket support

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [1]:
### MCP client implementation  - creating MCP client and connecting to MCP servers

# Step C1: Basic MCP Client
# client.py

import asyncio
from mcp import ClientSession
from mcp.client.stdio import stdio_client

## connect to mcp server which are stdio based
from mcp.client.stdio import stdio_client

client = stdio_client("python weather_stdio_server.py")

tools = client.list_tools()

result = client.call_tool(
    "get_weather",
    {"city": "Mumbai"}
)

print(result)



## connect to mcp server which are http based
from mcp.client.http import http_client

client = http_client("http://localhost:8000")

tools = client.list_tools()

result = client.call_tool(
    "get_weather",
    {"city": "Mumbai"}
)

print(result)



## connect to web socket mcp server
from mcp.client.websocket import websocket_client

client = websocket_client("ws://localhost:8000")

tools = client.list_tools()

result = client.call_tool("get_weather", {"city": "Mumbai"})

In [ ]:
#                  ┌─────────────────┐
#                  │      Agent      │
#                  │                 │
#                  │  MCPClientMgr   │
#                  └────────┬────────┘
#                           │
#         ┌─────────────────┼─────────────────┐
#         │                 │                 │
#         ▼                 ▼                 ▼
#   Internal MCP      External MCP      External MCP
#    (stdio)            (HTTP)           (WebSocket)
#         │                 │                 │
#         ▼                 ▼                 ▼
#     Local tools       Cloud tools      SaaS tools


# Step 1: Define MCP Client Wrapper Classes

# We unify stdio and network MCP clients under one interface.

# mcp_clients.py

from mcp.client.stdio import stdio_client
from mcp.client.http import http_client
from mcp.client.websocket import websocket_client


class BaseMCPClient:
    """Common interface for all MCP clients"""

    def list_tools(self):
        raise NotImplementedError

    def call_tool(self, name, arguments):
        raise NotImplementedError


# Internal MCP client (stdio)
class StdioMCPClient(BaseMCPClient):

    def __init__(self, command):
        """
        command example:
        python weather_stdio_server.py
        """
        self.client = stdio_client(command)

    def list_tools(self):
        return self.client.list_tools()

    def call_tool(self, name, arguments):
        return self.client.call_tool(name, arguments)


# External MCP client (HTTP)
class HTTPMCPClient(BaseMCPClient):

    def __init__(self, url):
        """
        example url:
        http://localhost:8000
        """
        self.client = http_client(url)

    def list_tools(self):
        return self.client.list_tools()

    def call_tool(self, name, arguments):
        return self.client.call_tool(name, arguments)


# External MCP client (WebSocket)
class WebSocketMCPClient(BaseMCPClient):

    def __init__(self, url):
        """
        example:
        ws://localhost:8000
        """
        self.client = websocket_client(url)

    def list_tools(self):
        return self.client.list_tools()

    def call_tool(self, name, arguments):
        return self.client.call_tool(name, arguments)
    

# Step 2: MCP Client Manager (handles multiple MCP servers)
# mcp_client_manager.py

class MCPClientManager:

    def __init__(self):
        self.clients = []
        self.tool_map = {}

    def register_client(self, client):

        self.clients.append(client)

        tools = client.list_tools()

        for tool in tools:
            tool_name = tool.name
            self.tool_map[tool_name] = client

    def list_all_tools(self):

        return list(self.tool_map.keys())

    def call_tool(self, tool_name, arguments):

        if tool_name not in self.tool_map:
            raise Exception(f"Tool {tool_name} not found")

        client = self.tool_map[tool_name]

        return client.call_tool(tool_name, arguments)

# Step 3: Agent Class integrating MCP Clients
# agent.py

class Agent:

    def __init__(self, llm, mcp_manager):

        self.llm = llm
        self.mcp = mcp_manager

    def build_prompt(self, query):

        tools = self.mcp.list_all_tools()

        prompt = f"""
        User Query: {query}

        Available tools:
        {tools}

        If needed, call tool using format:
        TOOL_CALL: tool_name(args)
        """

        return prompt

    def parse_tool_call(self, response):

        if "TOOL_CALL:" not in response:
            return None

        # Example:
        # TOOL_CALL: get_weather(city="Mumbai")

        part = response.split("TOOL_CALL:")[1].strip()

        name = part.split("(")[0]

        args_str = part.split("(")[1].split(")")[0]

        args = {}

        if args_str:
            key, value = args_str.split("=")
            args[key.strip()] = value.strip().replace('"', '')

        return name, args

    def run(self, query):

        prompt = self.build_prompt(query)

        llm_response = self.llm(prompt)

        tool_call = self.parse_tool_call(llm_response)

        if tool_call:

            tool_name, args = tool_call

            tool_result = self.mcp.call_tool(tool_name, args)

            final_prompt = f"""
            Tool Result:
            {tool_result}

            Answer user question.
            """

            return self.llm(final_prompt)

        return llm_response

In [ ]:
# Below is a **production-style Python implementation** showing:

# * Single Agent class
# * One MCP client manager
# * Multiple MCP servers
# * Tool discovery from MCP servers
# * Passing tool schemas to LLM
# * LLM deciding tool calls
# * MCP client executing tools
# * Returning result to LLM

# This is the **correct architecture used in real MCP agents**.

# ---

# # Architecture Overview

# ```
# User Query
#    ↓
# Agent
#    ↓
# MCP Client Manager
#    ↓
# Multiple MCP Servers
#    ↓
# Tools
#    ↓
# LLM decides tool
#    ↓
# MCP Client executes tool
#    ↓
# Result returned to Agent
# ```

# ---

# # Project Structure

# ```
# project/
# │
# ├── agent.py
# ├── mcp_client_manager.py
# ├── llm.py
# ├── weather_server.py
# ├── math_server.py
# ```

# ---

# # Step 1: Example MCP Servers

# ## weather_server.py

# ```python
# from mcp.server.fastmcp import FastMCP

# mcp = FastMCP("Weather Server")

# @mcp.tool()
# def get_weather(city: str) -> str:
#     """Get weather of a city"""
#     return f"Weather in {city} is 27°C"

# if __name__ == "__main__":
#     mcp.run()
# ```

# ---

# ## math_server.py

# ```python
# from mcp.server.fastmcp import FastMCP

# mcp = FastMCP("Math Server")

# @mcp.tool()
# def add(a: int, b: int) -> int:
#     """Add two numbers"""
#     return a + b

# if __name__ == "__main__":
#     mcp.run()
# ```

# ---

# # Step 2: MCP Client Manager

# This manages multiple MCP servers.

# ## mcp_client_manager.py

# ```python
# import asyncio
# from mcp import ClientSession
# from mcp.client.stdio import stdio_client


# class MCPClientManager:

#     def __init__(self, servers):
#         """
#         servers = [
#             {"command": "python", "args": ["weather_server.py"]},
#             {"command": "python", "args": ["math_server.py"]}
#         ]
#         """
#         self.servers = servers
#         self.sessions = []
#         self.tools = []

#     async def connect(self):

#         for server in self.servers:

#             read, write = await stdio_client(
#                 command=server["command"],
#                 args=server["args"]
#             )

#             session = ClientSession(read, write)

#             await session.initialize()

#             self.sessions.append(session)

#         await self.discover_tools()

#     async def discover_tools(self):

#         self.tools = []

#         for session in self.sessions:

#             tools = await session.list_tools()

#             for tool in tools:

#                 self.tools.append({
#                     "name": tool.name,
#                     "description": tool.description,
#                     "input_schema": tool.inputSchema,
#                     "session": session
#                 })

#     def get_tools_for_llm(self):

#         """Return tool schemas without session"""
#         return [
#             {
#                 "name": tool["name"],
#                 "description": tool["description"],
#                 "parameters": tool["input_schema"]
#             }
#             for tool in self.tools
#         ]

#     async def call_tool(self, tool_name, arguments):

#         for tool in self.tools:

#             if tool["name"] == tool_name:

#                 session = tool["session"]

#                 result = await session.call_tool(
#                     tool_name,
#                     arguments=arguments
#                 )

#                 return result

#         raise Exception("Tool not found")
# ```

# ---

# # Step 3: LLM Wrapper

# This simulates LLM deciding tool usage.

# ## llm.py

# Replace with OpenAI / any LLM.

# ```python
# import json


# class FakeLLM:

#     def generate(self, query, tools):

#         print("\nLLM received tools:")
#         for tool in tools:
#             print(tool["name"])

#         # Example decision logic
#         if "weather" in query.lower():

#             return {
#                 "tool": "get_weather",
#                 "arguments": {"city": "Mumbai"}
#             }

#         if "add" in query.lower():

#             return {
#                 "tool": "add",
#                 "arguments": {"a": 5, "b": 10}
#             }

#         return {
#             "final_answer": "No tool needed"
#         }
# ```

# ---

# # Step 4: Agent Class

# This is the main agent.

# ## agent.py

# ```python
# import asyncio
# from mcp_client_manager import MCPClientManager
# from llm import FakeLLM


# class Agent:

#     def __init__(self, mcp_client_manager, llm):

#         self.mcp = mcp_client_manager
#         self.llm = llm

#     async def initialize(self):

#         await self.mcp.connect()

#     async def run(self, user_query):

#         print("\nUser Query:", user_query)

#         # Step 1: Get tools from MCP
#         tools = self.mcp.get_tools_for_llm()

#         # Step 2: Send tools + query to LLM
#         llm_response = self.llm.generate(user_query, tools)

#         print("\nLLM Response:", llm_response)

#         # Step 3: If tool call needed
#         if "tool" in llm_response:

#             tool_name = llm_response["tool"]
#             arguments = llm_response["arguments"]

#             print(f"\nCalling tool: {tool_name}")

#             # Step 4: Call MCP client → server
#             result = await self.mcp.call_tool(
#                 tool_name,
#                 arguments
#             )

#             print("\nTool Result:", result)

#             return result

#         return llm_response["final_answer"]


# # MAIN
# async def main():

#     servers = [
#         {"command": "python", "args": ["weather_server.py"]},
#         {"command": "python", "args": ["math_server.py"]}
#     ]

#     mcp_manager = MCPClientManager(servers)

#     llm = FakeLLM()

#     agent = Agent(mcp_manager, llm)

#     await agent.initialize()

#     result = await agent.run("What is weather in Mumbai?")

#     print("\nFinal Answer:", result)


# asyncio.run(main())
# ```

# ---

# # Execution Flow

# Run:

# ```bash
# python agent.py
# ```

# Output:

# ```
# User Query: What is weather in Mumbai?

# LLM received tools:
# get_weather
# add

# LLM Response:
# {
#  "tool": "get_weather",
#  "arguments": {"city": "Mumbai"}
# }

# Calling tool: get_weather

# Tool Result:
# Weather in Mumbai is 27°C

# Final Answer:
# Weather in Mumbai is 27°C
# ```

# ---

# # Important Conceptual Flow

# Step 1
# Agent → MCP client manager → gets tools from all servers

# Step 2
# Agent → LLM

# Prompt contains:

# ```
# User query
# +
# Tool schemas
# ```

# Step 3
# LLM decides:

# ```
# tool: get_weather
# arguments: city=Mumbai
# ```

# Step 4
# Agent → MCP client manager

# Step 5
# MCP client manager → correct MCP server

# Step 6
# MCP server executes tool

# Step 7
# Result returned to agent

# ---

# # Real Production Upgrade (Using OpenAI)

# Replace FakeLLM with:

# ```python
# from openai import OpenAI

# client = OpenAI()

# response = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[...],
#     tools=tools
# )
# ```

# ---

# # Final Architecture Summary

# ```
# Agent
#  ├── LLM
#  └── MCP Client Manager
#       ├── MCP Server 1 (weather)
#       └── MCP Server 2 (math)
# ```

# Agent does NOT know:

# * API endpoints
# * tool implementations

# Only MCP handles that.

# ---

# # If you want next, I can show:

# Full production version with:

# • OpenAI function calling
# • Automatic tool selection
# • Multi-step reasoning agent

# This is how Cursor, Claude Desktop, etc work internally.


#### email responding agent

now i want to build an agent which will read the emails when it comes and drafts a reply and asks for my approval ....if i approave it will send that email

for this we require to integrate with email

real time email notification to my agent using event driven architecture, after that it consumes it and drafts a mail using email drafter tool and asks for my approval, it i approve it will send reply 

Here are the **exact official links** you need to set up Gmail API in Google Cloud Console:

---

## 🔹 1️⃣ Open Google Cloud Console

👉 [https://console.cloud.google.com/](https://console.cloud.google.com/)

---

## 🔹 2️⃣ Create a New Project

Direct link to project creation:

👉 [https://console.cloud.google.com/projectcreate](https://console.cloud.google.com/projectcreate)

After creating it, make sure the new project is selected (top navbar project dropdown).

---

## 🔹 3️⃣ Enable Gmail API

Direct link to Gmail API page:

👉 [https://console.cloud.google.com/apis/library/gmail.googleapis.com](https://console.cloud.google.com/apis/library/gmail.googleapis.com)

Click **Enable**

---

## 🔹 4️⃣ Go to Credentials Page

👉 [https://console.cloud.google.com/apis/credentials](https://console.cloud.google.com/apis/credentials)

---

## 🔹 5️⃣ Create OAuth 2.0 Client ID

Direct link:

👉 [https://console.cloud.google.com/apis/credentials/oauthclient](https://console.cloud.google.com/apis/credentials/oauthclient)

If prompted:

* Click **Configure Consent Screen** first
  Direct link:
  👉 [https://console.cloud.google.com/apis/credentials/consent](https://console.cloud.google.com/apis/credentials/consent)

---

## 🔹 6️⃣ When Creating OAuth Client

Choose:

* Application type → **Desktop app**
* Name → Anything (e.g., Gmail MCP Test)
* Click **Create**

---

## 🔹 7️⃣ Download credentials.json

After creation:

* Click **Download JSON**
* Save as `credentials.json`
* Place in your Python project folder

---

# ✅ That’s It

After this, you can run your Python OAuth script and it will open browser login.

---

If you get stuck at the **Consent Screen step**, tell me — that’s where most people get confused, and I’ll guide you through it cleanly.


Great 👍 since you already have `credentials.json`, let’s build your **Gmail Agent (Python + MCP-style tools)** cleanly with proper structure.

I’ll give you:

* ✅ Clean folder structure
* ✅ Auth handling
* ✅ Gmail service layer
* ✅ Tools (send, read, list, search)
* ✅ Agent entrypoint

---

# 📁 Recommended Folder Structure

```id="o2f8sj"
gmail-agent/
│
├── credentials.json        # downloaded from Google Cloud
├── token.pickle            # auto-created after first login
├── requirements.txt
│
├── app/
│   ├── __init__.py
│   ├── config.py
│   ├── auth.py
│   ├── gmail_service.py
│   ├── tools.py
│   └── agent.py
│
└── main.py
```

---

# 1️⃣ Install Requirements

Create `requirements.txt`

```id="m7a91v"
google-api-python-client
google-auth-httplib2
google-auth-oauthlib
```

Install:

```bash id="6r1dpo"
pip install -r requirements.txt
```

---

# 2️⃣ config.py

`app/config.py`

```python
SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.send",
    "https://www.googleapis.com/auth/gmail.modify",
]

CREDENTIALS_FILE = "credentials.json"
TOKEN_FILE = "token.pickle"
```

---

# 3️⃣ auth.py (OAuth Login + Token Storage)

`app/auth.py`

```python
import os
import pickle
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from .config import SCOPES, CREDENTIALS_FILE, TOKEN_FILE


def get_gmail_service():
    creds = None

    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file(
            CREDENTIALS_FILE, SCOPES
        )
        creds = flow.run_local_server(port=0)

        with open(TOKEN_FILE, "wb") as token:
            pickle.dump(creds, token)

    service = build("gmail", "v1", credentials=creds)
    return service
```

---

# 4️⃣ gmail_service.py (Core Gmail Logic)

`app/gmail_service.py`

```python
import base64
from email.message import EmailMessage


class GmailService:
    def __init__(self, service):
        self.service = service

    def send_email(self, to, subject, body):
        message = EmailMessage()
        message.set_content(body)
        message["To"] = to
        message["Subject"] = subject

        encoded_message = base64.urlsafe_b64encode(
            message.as_bytes()
        ).decode()

        self.service.users().messages().send(
            userId="me",
            body={"raw": encoded_message}
        ).execute()

        return "Email sent successfully"

    def list_emails(self, max_results=10):
        results = self.service.users().messages().list(
            userId="me",
            maxResults=max_results
        ).execute()

        messages = results.get("messages", [])
        email_list = []

        for msg in messages:
            msg_data = self.service.users().messages().get(
                userId="me",
                id=msg["id"]
            ).execute()

            headers = msg_data["payload"]["headers"]
            subject = next(
                (h["value"] for h in headers if h["name"] == "Subject"),
                "No Subject"
            )

            email_list.append({
                "id": msg["id"],
                "subject": subject
            })

        return email_list

    def search_emails(self, query, max_results=5):
        results = self.service.users().messages().list(
            userId="me",
            q=query,
            maxResults=max_results
        ).execute()

        return results.get("messages", [])

    def read_email(self, message_id):
        msg = self.service.users().messages().get(
            userId="me",
            id=message_id,
            format="full"
        ).execute()

        return msg
```

---

# 5️⃣ tools.py (MCP-style Tool Layer)

`app/tools.py`

```python
from .gmail_service import GmailService


class GmailTools:
    def __init__(self, service):
        self.gmail = GmailService(service)

    def send_email(self, args):
        return self.gmail.send_email(
            args["to"],
            args["subject"],
            args["body"]
        )

    def list_emails(self, args):
        return self.gmail.list_emails(
            args.get("max_results", 10)
        )

    def search_emails(self, args):
        return self.gmail.search_emails(
            args["query"],
            args.get("max_results", 5)
        )

    def read_email(self, args):
        return self.gmail.read_email(
            args["message_id"]
        )
```

---

# 6️⃣ agent.py (Agent Controller)

`app/agent.py`

```python
from .auth import get_gmail_service
from .tools import GmailTools


class GmailAgent:
    def __init__(self):
        service = get_gmail_service()
        self.tools = GmailTools(service)

    def run(self):
        print("Gmail Agent Ready")
        print("1. List Emails")
        print("2. Send Email")

        choice = input("Choose: ")

        if choice == "1":
            emails = self.tools.list_emails({})
            for e in emails:
                print(e)

        elif choice == "2":
            to = input("To: ")
            subject = input("Subject: ")
            body = input("Body: ")

            result = self.tools.send_email({
                "to": to,
                "subject": subject,
                "body": body
            })

            print(result)
```

---

# 7️⃣ main.py (Entry Point)

```python
from app.agent import GmailAgent

if __name__ == "__main__":
    agent = GmailAgent()
    agent.run()
```

---

# 🚀 Run Your Agent

```bash
python main.py
```

First time:

* Browser opens
* Login to Gmail
* token.pickle created

After that:

* No login needed

---

# 🎯 What You Now Have

✔ Clean architecture
✔ OAuth handling
✔ Send emails
✔ List inbox
✔ Search emails
✔ Agent wrapper
✔ Easy to convert to MCP server




after running code browser opens and when you select your email ...access denied comes

Error 403: access_denied

"App has not completed Google verification"

This happens because your OAuth app is in Testing mode inside Google Cloud Console.

🔥 Why This Happens

When you create OAuth credentials:

The app is set to Testing

Only approved test users can log in

Your Gmail account is NOT added as a test user

So Google blocks it.

Step 1 — Go to OAuth Consent Screen

Direct link:

👉 https://console.cloud.google.com/apis/credentials/consent

Make sure correct project is selected at top.

👉 Click Audience

That is where Test Users now lives.

Direct link (same page):

https://console.cloud.google.com/auth/audience

✅ Inside “Audience” Page

You will see:

User Type (Internal / External)

Publishing Status

Test Users section

🔥 What To Check
1️⃣ User Type

It must be:

External

If it says Internal, that’s the problem.

2️⃣ Publishing Status

It should say:

Testing

3️⃣ Test Users Section

Scroll down and click:

+ Add users

Add:

viishhnupancharatnela@gmail.com

Click Save.


# 🚀 Project: Autonomous Research & Insight Engine

This agent:

* Monitors new research daily
* Identifies emerging trends
* Detects research gaps
* Compares SOTA methods
* Suggests experiment ideas
* Generates experiment plans
* Optionally runs experiments
* Logs everything

This is research-level automation.

---

# 🧠 High-Level Goal

Input:

> “Track progress in LLM efficiency and suggest experiment ideas.”

Output:

* Latest relevant papers
* Trend analysis
* Technique clustering
* Identified research gaps
* Proposed experiment setup
* Evaluation plan

That’s impressive.

---

# 🧩 MCP Servers You Should Build

We’ll use real-world APIs + custom intelligence tools.

---

## 1️⃣ Research Search MCP Server

Use:

* arXiv API
* Semantic Scholar API

Tools:

* search_papers(query, days_filter)
* get_paper_details(paper_id)
* download_pdf(paper_id)

---

## 2️⃣ PDF Intelligence MCP Server

Tools:

* extract_text_from_pdf
* extract_method_section
* extract_experiments
* extract_metrics
* extract_limitations

You can optionally plug in a vision model for PDF layout understanding.

---

## 3️⃣ Research Memory MCP Server

Vector DB:

* store_paper_embedding
* search_similar_papers
* cluster_papers
* detect_topic_shift

This allows trend detection over time.

---

## 4️⃣ Analytical LLM MCP Server

Specialized prompts:

* compare_methods
* extract_novelty
* detect_research_gap
* risk_assessment
* propose_experiment

You can bind a cheaper LLM here for structured reasoning.

---

## 5️⃣ Experiment Design MCP Server

Tools:

* generate_experiment_plan
* generate_hyperparameters
* suggest_dataset
* generate_evaluation_metrics
* estimate_compute_cost

---

## 6️⃣ Optional: ML Execution MCP Server

If you want insane level:

* run_experiment
* log_results
* compare_runs

You can integrate:

* MLflow
* Local training scripts

Now it becomes autonomous experimentation.

---

# 🧠 Multi-Agent Architecture

We design real multi-agent collaboration.

---

## 🧠 1️⃣ Orchestrator Agent

Role:

* Understand research goal
* Break into subtasks
* Assign agents
* Aggregate outputs
* Maintain research memory

This is the brain.

---

## 🔎 2️⃣ Discovery Agent

Role:

* Search new papers
* Filter relevance
* Rank by impact
* Detect citations

Uses:

* arXiv server
* Semantic Scholar server

---

## 📄 3️⃣ Paper Analysis Agent

Role:

* Extract contributions
* Identify architecture
* Extract training details
* Identify limitations
* Extract metrics

Uses:

* PDF MCP
* Analytical LLM tool

---

## 📊 4️⃣ Trend Intelligence Agent

Role:

* Cluster papers
* Detect emerging themes
* Compare SOTA metrics
* Detect technique acceleration

Uses:

* Research Memory server
* Analytical LLM

---

## 🧪 5️⃣ Experiment Planner Agent

Role:

* Identify research gaps
* Propose new ideas
* Suggest experiments
* Define evaluation setup

This is where you shine.

---

## 🔬 6️⃣ (Optional) Execution Agent

Role:

* Generate training code
* Execute
* Log metrics
* Compare results

---

# 🔁 Example Full Flow

User:

> “Monitor diffusion model efficiency improvements.”

System:

1. Discovery Agent:

   * Fetch last 7 days arXiv papers

2. Analysis Agent:

   * Extract architectures
   * Extract training tricks
   * Extract reported efficiency gains

3. Trend Agent:

   * Cluster by approach (quantization, pruning, distillation)
   * Compare reported improvements

4. Planner Agent:

   * Identify gap:
     “No paper combined distillation + structured pruning for diffusion UNets.”

5. Experiment Agent:

   * Propose experiment
   * Suggest dataset
   * Suggest baseline
   * Define metrics

Output:

```text
Emerging Trend:
- 4 papers focus on post-training quantization

Gap Identified:
- No evaluation of hybrid compression pipelines

Proposed Experiment:
- Train baseline UNet
- Apply KD + structured pruning
- Measure FID, throughput, VRAM usage
```

This is not normal.

---

# 🧠 Advanced Mode: Continuous Monitoring

Run daily:

* Fetch new papers
* Update vector DB
* Re-cluster topics
* Detect new trend acceleration

You now have:

> Research Radar System

---

# 🧠 Even More Advanced: Hypothesis Generator

Train it to detect:

* Contradictions between papers
* Under-explored parameter spaces
* Missing ablation studies

That’s serious research augmentation.

---

# 🏗 Suggested Folder Structure

```text
research_agent/
│
├── mcp_servers/
│   ├── arxiv_server.py
│   ├── pdf_server.py
│   ├── memory_server.py
│   ├── analytical_llm_server.py
│   ├── experiment_server.py
│
├── agents/
│   ├── orchestrator.py
│   ├── discovery_agent.py
│   ├── analysis_agent.py
│   ├── trend_agent.py
│   ├── planner_agent.py
│
├── memory/
│   ├── vector_store.py
│
├── agent.py
```

---

# 🧠 Model Specialization (Very Important)

Instead of one LLM:

* Large reasoning model → Orchestrator
* Cheap model → Paper extraction
* Structured reasoning model → Experiment planning

You can bind each as MCP tools.

That shows system-level thinking.

---

# 🔥 Why This Project Is Elite

Because it demonstrates:

* Tool orchestration
* Multi-agent collaboration
* Vector memory
* Trend detection
* Research reasoning
* Experiment design
* Optional ML execution
* API integration
* Real domain depth

This is portfolio gold.

---

# 🚀 If You Want Next Step

I can:

1. Design exact MCP tool schemas
2. Write one full MCP server (arXiv)
3. Design Orchestrator logic
4. Show multi-agent delegation loop
5. Design research gap detection prompt
6. Design experiment planning pipeline

